In [1]:
!nvidia-smi

Mon May  4 14:23:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [3]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.7.0
Uninstalling transformers-5.7.0:
  Successfully uninstalled transformers-5.7.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.7.0-py3-none-any.whl (10.5 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)


In [4]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt")
import torch
from tqdm import tqdm


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [6]:
model_ckpt="google/pegasus-cnn_dailymail"
tokenizer=AutoTokenizer.from_pretrained(model_ckpt)
model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SyntaxError: invalid syntax (2025399527.py, line 1)

In [8]:
dataset_samsum=load_from_disk('samsum_dataset')
dataset_samsum

KeyboardInterrupt: 

In [ ]:
!wget https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
!unzip summarizer-data.zip


--2026-05-04 15:04:13--  https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip [following]
--2026-05-04 15:04:13--  https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7903594 (7.5M) [application/zip]
Saving to: ‘summarizer-data.zip.3’

summarizer-data.zip 100%[===================>]   7.54M  --.-KB/s    in 0.02s   

2026-05-04 15:04:14 (356 MB/s) - ‘summarizer-data.zip.3’ saved [7903

In [ ]:
yes A | unzip summarizer-data.zip
split_length=[len(dataset_samsum[split])for split in dataset_samsum]
print(f"split lenght {split_length}")
print(f"features {dataset_samsum['train'].column_names}")
print("\ndialogue:")
print(dataset_samsum["test"][1]["dialogue"])
print("\nsummary:")
print(dataset_samsum["test"][1]['summary'])


In [ ]:
def convert_examples_to_features(example_batch):
    input_encoding = tokenizer(
        example_batch['dialogue'],
        max_length=1024,
        truncation=True
    )

    target_encoding = tokenizer(
        text_target=example_batch['summary'],
        max_length=1024,
        truncation=True
    )

    return {
        'input_ids': input_encoding['input_ids'],
        'attention_mask': input_encoding['attention_mask'],
        'labels': target_encoding['input_ids']
    }

In [ ]:
dataset_samsum_pt=dataset_samsum.map(convert_examples_to_features,batched=True)

In [ ]:
dataset_samsum_pt['train']

In [ ]:
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator=DataCollatorForSeq2(tokenizer,model=model_pegasus)

In [ ]:
!pip install --upgrade transformers
from transformers import TrainingArguments,Trainer
trainer_args=TrainingArguments(
    output_dir='pegasus-samsum',num_train_epochs=1,warmup_steps=500,
    per_device_train_batch_size=1,per_device_eval_batch_size=1,
    weight_decay=0.01,logging_steps=10,
    evaluation_strategy="steps",eval_steps=500,save_steps=1e6,
    gradient_accumulation_steps=16
)


In [ ]:
trainer=Trainer(model=model_pegasus,
                args=trainer_agrs,
                tokenzer=tokenizer,
                data_collator=DataCollatorForSeq2Seq,
                train_dataset=dataset_samsum_pt['test'],
                train_dataset=dataset_samsum_pt['validation'])

In [ ]:
trainer.train()

In [ ]:
def generate_batch_sized_chunks(list_of_elements,batch_size):
  for i in range(len(list_of_elements),batch_size):
    yield list_of_elements[i:i+batch_size]

def calculate_metric_on_test_ds(dataset,metric,model,tokenizer,
                                batch_size=16,device=device,
                                column_text="article",
                                column_summary="highlights"):
  article_batches=list(generate_batch_sized_chunks(dataset[column_text],batch_size))
  target_batches=list(generate_batch_sized_chunks(dataset[column_summary],batch_size))
  for article_batch,target_batch in tqdm(
      zip(article_batches,target_batches),total=len(article_batches)):
    inputs=tokenizer(article_batch,max_length=1024,truncation=True,
                     padding="max_length",return_tensors='pt')
    summaries=model.generate(input_ids=inputs['input_ids'].to(device),
                             attention_mask=inputs['attention_mask'].to(device),
                             length_penalty=0.8,num_beams=8,max_length=128)


    decoded_summaries=[d.replace(""," ") for d in decoded_summaries]
    metric.add_batch(predictions=decoded_summaries,references=target_batch)

  score=metric.compute()
  return score


In [ ]:
rouge_names=['rouge1','rouge2','rougeL','rougeLsum']
rouge_metric=load_metric('rouge')

In [ ]:
score=calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10],rouge_metric,trainer.model,tokenizer,batch_size=2,column_text='dialogue',column_summary='summary'
)
rouge_dict=dict((rn,score(rn).mid.fmeasure) for rn in rouge_names)
pd.dataframe(rouge_dict,index=[f'pegasus'])

In [ ]:
model_pegasus.save_pretrained("pegasus-samsum-model")

In [ ]:
tokenizer.save_pretrained("tokenizer")


In [ ]:
tokenizer=AutoTokenizer.from_pretrained("/content/tokenizer")

In [ ]:
gen_kwargs={"length_penalty":0.8,"num_beams":8,"max_length":128}

sample_text=dataset_samsum["test"][0]["dialogue"]
reference=dataset_samsum["test"][0]["summary"]
pipe=pipeline("summarization",model="pegasus-samsum-model",tokenizer=tokenizer)
print("Dialogue:")
print(sample_text)
print("\nReference Summary:")
print(reference)
print("\nModel Summary:")
print(pipe(sample_text,**gen_kwargs)[0]["summary_text"])